In [1]:
import sys

sys.path.insert(0, "../src")

from dotenv import load_dotenv

load_dotenv("../.env.local")

from tqdm import tqdm
from fsparts_scraper.crawler import (
    fetch_sitemap,
    parse_sitemap_xml,
    filter_product_urls,
    KNOWN_CATEGORY_SLUGS,
    save_urls,
    load_urls,
)
from fsparts_scraper.extractor import (
    fetch_product_html,
    extract_prices,
    extract_images,
    clean_html,
    close_browser,
)
from fsparts_scraper.llm import get_client as get_llm_client, extract_product_data
from fsparts_scraper.images import process_product_images
from fsparts_scraper.supabase_client import get_client, upsert_product, make_slug
from fsparts_scraper.checkpoint import (
    save_checkpoint,
    load_checkpoint,
    list_checkpoints,
)

llm_client = get_llm_client()
print("NVIDIA NIM client ready")

NVIDIA NIM client ready


In [2]:
urls = load_urls()
if not urls:
    print("Fetching sitemap...")
    xml = fetch_sitemap()
    all_urls = parse_sitemap_xml(xml)
    urls = filter_product_urls(all_urls, KNOWN_CATEGORY_SLUGS)
    save_urls(urls)
    print(f"Saved {len(urls)} product URLs to data/urls.txt")
else:
    print(f"Loaded {len(urls)} URLs from data/urls.txt")

Loaded 198 URLs from data/urls.txt


In [3]:
errors = []
seen_urls = {c.get("source_url") for c in list_checkpoints()}

for url in tqdm(urls, desc="Extracting"):
    sku_slug = url.rstrip("/").split("/")[-1]
    if url in seen_urls:
        continue

    try:
        html = await fetch_product_html(url)
        prices = extract_prices(html)
        images = extract_images(html)
        cleaned = clean_html(html)
        llm_data = extract_product_data(cleaned, client=llm_client)

        if llm_data is None:
            save_checkpoint(
                sku_slug,
                {"sku": sku_slug, "source_url": url, "status": "extraction_failed"},
            )
            errors.append(url)
            continue

        sku = llm_data.get("sku") or sku_slug
        checkpoint = {
            **llm_data,
            "sku": sku,
            "slug": make_slug(llm_data.get("name", sku)),
            "source_url": url,
            "image_urls_original": images,
            "images": [],
            "status": "extracted",
            **prices,
        }
        save_checkpoint(sku, checkpoint)
        seen_urls.add(url)

    except Exception as e:
        print(f"  ERROR on {url}: {e}")
        errors.append(url)

print(f"\nDone. Errors: {len(errors)}")
for u in errors:
    print(f"  {u}")

Extracting: 100%|██████████| 198/198 [25:13<00:00,  7.64s/it]


Done. Errors: 0


In [4]:
sb = get_client()
for cp in tqdm(list_checkpoints(status="extracted"), desc="Uploading images"):
    try:
        public_urls = process_product_images(cp, sb)
        cp["images"] = public_urls
        cp["status"] = "images_done"
        save_checkpoint(cp["sku"], cp)
    except Exception as e:
        print(f"  ERROR on {cp['sku']}: {e}")

Uploading images: 100%|██████████| 170/170 [06:09<00:00,  2.18s/it]


In [5]:
sb = get_client()
upserted = 0
for cp in tqdm(list_checkpoints(status="images_done"), desc="Upserting"):
    try:
        upsert_product(cp, sb)
        cp["status"] = "upserted"
        save_checkpoint(cp["sku"], cp)
        upserted += 1
    except Exception as e:
        print(f"  ERROR on {cp['sku']}: {e}")

print(f"\n{upserted} products upserted to Supabase")

Upserting: 100%|██████████| 170/170 [00:33<00:00,  5.08it/s]


170 products upserted to Supabase


In [6]:
from collections import Counter

statuses = Counter(c["status"] for c in list_checkpoints())
print("Status summary:")
for status, count in sorted(statuses.items()):
    print(f"  {status}: {count}")
print(f"  Total: {sum(statuses.values())}")

Status summary:
  extracted: 5
  upserted: 170
  Total: 175


In [7]:
close_browser()